In [2]:
import sys
sys.path.append('/home/nathan/thermo_flux')

import pandas as pd
import os
# Add the parent directory to sys.path
sys.path.append(os.path.abspath('../3_src/'))
# Now import Model from freeflux

import numpy as np

import seaborn as sns
import analysis.io_func as io_func
import matplotlib.pyplot as plt

from analysis import config_13c as config
import json

/home/nathan/PhD/04.Computational/202601_13C/3_src/freeflux/src/freeflux/core/reaction.py:104: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.substrates_info = pd.concat((self.substrates_info, newSubs))
/home/nathan/PhD/04.Computational/202601_13C/3_src/freeflux/src/freeflux/core/reaction.py:132: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.products_info = pd.concat((self.products_info, newPros))
/home/nathan/PhD/04.Computational/202601_13C/3_src/freeflux/src/freeflux/core/reaction.py:

###  i first scan through rxn common to my yeast model and y9 in preivous notebook 4c1. 
then here plot for reactions that have different directions

In [3]:
# Import general configs:
palette ,cpalette,cfg=io_func.import_plotconfigs(cpalette=True)

In [4]:
condition='WT' 
model=config.model
bootstrap_flux_dfwt=config.bootstrap_flux_df(condition)#pd.read_csv(f'{config.DRAWNSAMPLES_FOLDER}/{condition}_drawn_samples.csv',index_col=0)


In [5]:
condition='TM6' 
model=config.model
bootstrap_flux_dftm6=config.bootstrap_flux_df(condition)#pd.read_csv(f'{config.DRAWNSAMPLES_FOLDER}/{condition}_drawn_samples.csv',index_col=0)


In [6]:
df_rxn_dir = pd.DataFrame(index=bootstrap_flux_dfwt.index)

for r in bootstrap_flux_dfwt.index:

    # WT
    values_wt = bootstrap_flux_dfwt.loc[r].values

    direc = 1 if np.all(values_wt >= 0) else 0
    direc = -1 if np.all(values_wt <= 0) else direc

    if direc == 1:
        df_rxn_dir.loc[r, 'WT'] = 'Forward'
    elif direc == -1:
        df_rxn_dir.loc[r, 'WT'] = 'Backward'
    else:
        df_rxn_dir.loc[r, 'WT'] = 'Reversible'

    # TM6
    values_tm6 = bootstrap_flux_dftm6.loc[r].values

    directm6 = 1 if np.all(values_tm6 >= 0) else 0
    directm6 = -1 if np.all(values_tm6 <= 0) else directm6

    if directm6 == 1:
        df_rxn_dir.loc[r, 'TM6'] = 'Forward'
    elif directm6 == -1:
        df_rxn_dir.loc[r, 'TM6'] = 'Backward'
    else:
        df_rxn_dir.loc[r, 'TM6'] = 'Reversible'

In [7]:
df_rxn_dir#.loc[['PItm-2+H','CO2t-1','CO2t-1+H']]#,accepted_samples].T.describe()

,WT,TM6
ASNS1,Forward,Forward
ASPK,Forward,Forward
ASPTA,Backward,Backward
ASPTAm,Reversible,Forward
ASAD,Forward,Forward
...,...,...
SUCCPItm-2-2,Backward,Backward
SUCFUMtm-2-2,Reversible,Reversible
THRtm+0,Forward,Forward
biomass_ce,Forward,Forward


In [8]:
# df_rxn_dir.to_csv(f'{config.RESULTS_FOLDER}/rxn_finaldirections_2conds.csv')

comparing reaction direction of WT, TM6 and yeast 9 

In [9]:
diff_dir=df_rxn_dir.copy()#[df_rxn_dir['WT']!=df_rxn_dir['TM6']].copy()#.index
## expand with ['ACOAH','OAAtm-2','PItm-2','SO4t-2']
for rid in ['OAAtm-2','PItm-2','SO4t-2']:
       diff_dir.loc[rid]=df_rxn_dir.loc[rid]
##manually curate for yeast 9 
diff_dir
y9_dirs={'ASPTAm':'Backward','FUM':'Reversible','FUMm':'Reversible','MDHm':'Reversible','h2o_EX':'Reversible','ATPS3m':'Forward','CO2t-1+H':'Reversible',
       'H2Ot+0':'Reversible','AKGMALtm-2-2':'Backward','ASPGLUtm-1-1':'Backward','CO2tm-1+H':'Reversible','H2Otm+0':'Reversible','PYRtm-1+H':'Forward',
       'OAAtm-2':'Forward','PItm-2':'Forward','SO4t-2':'Forward'
       }
diff_dir['Yeast9']=diff_dir.index.map(y9_dirs)


In [10]:
with open('reaction_direction_check_part2.json','r') as f:
    irr_rxn_dict_p2=json.load(f)

In [11]:
for rid,direc in irr_rxn_dict_p2.items():
    direc=int(direc)
    if rid in df_rxn_dir.index:
        # print(diff_dir.loc[rid],direc)
        if direc ==1 :
            diff_dir.loc[rid,'Yeast9']='Forward'
        elif direc== 0 :
            diff_dir.loc[rid,'Yeast9']='Backward'
        elif direc== 2 :
            diff_dir.loc[rid,'Yeast9']='Reversible'
# diff_dir.dropna(inplace=True)


## read from manually built json 
only irreverisble reactions. if 1 : forward, otherwise backward
i built it by looking at irr reactions and checking taht they were the same direction than us

In [12]:
with open('reaction_direction_check.json','r') as f:
    irr_rxn_dict=json.load(f)


## read previous identification (df built in previous notebook)

In [13]:
for rid,direc in irr_rxn_dict.items():
    direc=int(direc)
    if rid in df_rxn_dir.index:
        # print(diff_dir.loc[rid],direc)
        if direc ==1 :
            diff_dir.loc[rid,'Yeast9']='Forward'
        elif direc== 0 :
            diff_dir.loc[rid,'Yeast9']='Backward'
# diff_dir.dropna(inplace=True)


In [ ]:
# diff_dir.to_csv(f'{config.FREEFLUXRUN_FOLDER}/df_bootstrap_compare_reaction_directions.csv')

also load second part on rest of reactions

In [16]:
cond_dir_compare=diff_dir[['WT','TM6']]
cond_dir_compare

,WT,TM6
ASNS1,Forward,Forward
ASPK,Forward,Forward
ASPTA,Backward,Backward
ASPTAm,Reversible,Forward
ASAD,Forward,Forward
...,...,...
SUCCPItm-2-2,Backward,Backward
SUCFUMtm-2-2,Reversible,Reversible
THRtm+0,Forward,Forward
biomass_ce,Forward,Forward


In [17]:
cond_dir_compare.loc[['ALDD2ym','ICDHym']]

,WT,TM6
ALDD2ym,Forward,Forward
ICDHym,Forward,Forward


In [18]:
# index_same=cond_dir_compare[cond_dir_compare['WT']==cond_dir_compare['TM6']].index
# df_comp_flux=pd.DataFrame(full_flux_dftm6.loc[index_same,accepted_samplestm6].median(axis=1))
# df_comp_flux['WT']=full_flux_df.loc[index_same,accepted_samples].median(axis=1)
pd.set_option('display.max_rows', 20)
# df_comp_flux

In [19]:
def counts_diff(diff_dir, condition, return_index=True,only_wrong=True):
    # Select relevant columns
    diff_dir_cond = diff_dir[[condition, 'Yeast9']].copy()
    
    # Row-wise mask
    if only_wrong :
        mask = (
            (diff_dir_cond[condition] != diff_dir_cond['Yeast9']) &
            (diff_dir_cond[condition] != 'Reversible') &
            (diff_dir_cond['Yeast9'] != 'Reversible')
        )
    else:
        mask = (
            (diff_dir_cond[condition] != diff_dir_cond['Yeast9']) &
            (diff_dir_cond[condition] == 'Reversible')
        )

    
    if return_index:
        return diff_dir_cond.index[mask]
    else:
        return diff_dir_cond[mask]
    # return dfcount
index_wt_diff=counts_diff(diff_dir,'WT',only_wrong=True,return_index=False)
index_tm6_diff=counts_diff(diff_dir,'TM6',only_wrong=True,return_index=False)

In [20]:
df_table=index_wt_diff
df_table['TM6']=index_tm6_diff['TM6']
# df_table.loc['FTHFL','TM6']=df_rxn_dir.loc['FTHFL','TM6']
# df_table.loc['AKGMALtm-2-2','TM6']=df_rxn_dir.loc['AKGMALtm-2-2','TM6']

df_table#.shape


,WT,Yeast9,TM6
FTHFL,Backward,Forward,Backward
GLUDym,Backward,Forward,Backward
PHETA1,Backward,Forward,Backward
GLYCt+0,Forward,Backward,Forward
SO4t-2,Backward,Forward,Backward
2OXOADPAKGtm-2-2,Forward,Backward,Forward
AKGMALtm-2-2,Backward,Forward,Backward
OAAtm-2,Backward,Forward,Backward
ORNtm+1-H,Backward,Forward,Backward
PItm-2,Backward,Forward,Backward
